[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/26_lora.ipynb)

# 🟠 Medium: LoRA (Low-Rank Adaptation)

Implement **LoRA** — parameter-efficient fine-tuning for large models.

$$h = W_0 x + \frac{\alpha}{r} B A x$$

### Signature
```python
class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Requirements
- `self.linear`: frozen `nn.Linear` (weight & bias `requires_grad=False`)
- `self.lora_A`: `nn.Parameter(rank, in_features)` — random init
- `self.lora_B`: `nn.Parameter(out_features, rank)` — **zero** init
- Scaling: `alpha / rank`

### My notes:

#### `nn.Parameter` vs `nn.Linear`

- **`nn.Parameter`** is just a trainable tensor, so we must manually do the matrix multiplication in `forward()`, e.g. `x @ self.lora_A.T`.
- **`nn.Linear`** contains `nn.Parameter` weights internally and performs the matrix multiplication automatically when called, e.g. `self.lora_A(x)`.

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn as nn

In [37]:
# ✏️ YOUR IMPLEMENTATION HERE

class LoRALinear(nn.Module):
    def __init__(self, in_features, out_features, rank, alpha=1.0):
        super().__init__()
        # pass  # frozen linear + lora_A + lora_B
        self.linear = nn.Linear(in_features, out_features)
        self.linear.weight.requires_grad = False  # freeze the base linear layer
        self.linear.bias.requires_grad = False  # freeze the base linear layer bias
        # self.lora_A = nn.Linear(in_features, rank, bias=False)
        # self.lora_B = nn.Linear(rank, out_features, bias=False)
        self.lora_A = nn.Parameter(torch.randn(rank, in_features)) # with nn.Parameter, we have to do the matrix multiplication manually in forward
        self.lora_B = nn.Parameter(torch.zeros(out_features, rank)) # like x @ self.lora_A.T
        self.scaling = alpha / rank


    def forward(self, x):
        # pass  # base + lora
        # return self.linear(x) + self.scaling * self.lora_B(self.lora_A(x))
        return self.linear(x) + self.scaling * (x @ self.lora_A.T) @ self.lora_B.T

In [38]:
# 🧪 Debug
layer = LoRALinear(16, 8, rank=4)
x = torch.randn(2, 16)
print('Output:', layer(x).shape)
print('Trainable:', sum(p.numel() for p in layer.parameters() if p.requires_grad))
# print('lora_A params:', sum(p.numel() for p in layer.lora_A.parameters()))
# print('lora_B params:', sum(p.numel() for p in layer.lora_B.parameters()))
print('lora_A params:', layer.lora_A.numel())
print('lora_B params:', layer.lora_B.numel())
print('Base linear weight params:', layer.linear.weight.numel())
print('Base linear bias params:', layer.linear.bias.numel())
print('Total:    ', sum(p.numel() for p in layer.parameters()))

Output: torch.Size([2, 8])
Trainable: 96
lora_A params: 64
lora_B params: 32
Base linear weight params: 128
Base linear bias params: 8
Total:     232


In [39]:
# ✅ SUBMIT
from torch_judge import check
check('lora')


🧪 Testing: LoRA (Low-Rank Adaptation) (Medium)
──────────────────────────────────────────────────
  ✅ [1/5] Base weights frozen (0.5ms)
  ✅ [2/5] LoRA parameter shapes (0.1ms)
  ✅ [3/5] B=0 means output equals base (1.2ms)
  ✅ [4/5] Only LoRA params get gradients (0.3ms)
  ✅ [5/5] Forward computation (0.5ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (2.6ms total)
  Progress saved. Run status() to see your dashboard.

